# Data Analytics and Visualisation in Data-Centric Engineering

This notebook is your working copy for the workshop. Read each section header, then fill in the code cell below it to build up the analysis step by step.

The dataset is 90 days of 10-minute wind turbine telemetry for three turbines at a shared site. All three see the same regional weather, so any divergence between them is an asset signal rather than a weather effect.

Check that `turbine_telemetry.csv` is in the same folder as this notebook before starting.

If you get stuck on the Python syntax, open `snippets.ipynb` — it has copy-pasteable patterns for each step.

## How to use this notebook

Read each section header, then fill in the code cell below it. If a step is unfamiliar, check `snippets.ipynb` for the matching pattern and adapt it to the turbine columns. If you already know the pattern, write your own version directly and move on.

# Wind Turbine Telemetry Schema

## Dataset Grain

- One row per turbine per 10-minute timestamp.
- Three turbines are simulated: `WT-01`, `WT-02`, and `WT-03`.

## Column Dictionary

| Column | Type | Description | How it is used |
|---|---|---|---|
| `Timestamp` | datetime | 10-minute observation time for the record. | Used for time-series plots, trend fitting, rolling averages, and failure timing. |
| `Turbine_ID` | categorical | Turbine identifier, typically `WT-01`, `WT-02`, or `WT-03`. | Used to separate the turbine data |
| `Ambient_Temperature_C` | float | Local ambient temperature around the turbine nacelle area. | Environmental Variable |
| `Macro_Wind_Speed_mps` | float | Shared regional wind speed before local wake and gust effects. | The broader wind environment. |
| `Wind_Speed_mps` | float | Working wind speed experienced by the turbine after local effects. | Directly drives RPM and power output; distinct from the shared regional `Macro_Wind_Speed_mps`. |
| `Wind_Gust_mps` | float | Short-term gust component added to the wind field. | Used to explore links with RPM and transient loading. |
| `Local_Wake_Turbulence_mps` | float | Local wake-induced turbulence component around each turbine. | Helps explain turbine-to-turbine differences in effective wind loading. |
| `Effective_Wind_Speed_mps` | float | Combined wind speed after macro wind, gust, and local wake effects. | Operating wind variable. |
| `Turbulence_Intensity` | float | Dimensionless turbulence metric derived from wind and gust variability. | Operating wind variable |
| `Atmospheric_Pressure_hPa` | float | Local atmospheric pressure. | Weather context variable |
| `Relative_Humidity_pct` | float | Relative humidity percentage. | Weather context variable |
| `Wind_Direction_deg` | float | Wind direction in degrees from 0 to 360. | Used in yaw error plots and directiona
| `Bearing_Temperature_C` | float | Bearing temperature in degrees Celsius. | Bearing health signal. |
| `Vibration_mm_s` | float | Vibration magnitude in mm/s. | Key condition-monitoring signal |
| `RPM` | float | Rotor rotational speed. | Sensor variable. |
| `Power_kW` | float | Electrical power output in kilowatts. | Sensor variable. |
| `Turbine_Load_Percent` | float | Power output scaled as a percentage of rated power. | Normalised load view for interpretation and comparison. |
| `Yaw_Error_deg` | float | Difference between wind direction and turbine yaw alignment. | Used in yaw misalignment exploration. |
| `Blade_Pitch_Deg` | float | Blade pitch angle in degrees. | Used in pitch-vs-power relationships and curtailment checks. |
| `Generator_Current_A` | float | Generator current in amps. | Electrical response variable that follows output changes. |
| `Nacelle_Temperature_C` | float | Nacelle temperature in degrees Celsius. | Secondary thermal indicator. |
| `Oil_Pressure_bar` | float | Hydraulic or lubrication oil pressure in bar. | Mechanical health indicator for drivetrain condition. |
| `Gearbox_Temperature_C` | float | Gearbox temperature in degrees Celsius. | Secondary drivetrain thermal health indicator. |
| `Operational_State` | categorical | Simulated operating mode such as `healthy`, `degrading`, `drifting`, or `offline`. | Useful for labels, filtering, and interpretation. |
| `Maintenance_Flag` | categorical | Human-readable maintenance condition such as `none`, `bearing degradation`, `catastrophic seizure`, or `early drift`. | Connects telemetry patterns to maintenance meaning. |
| `Grid_Priority` | categorical | Simplified grid dispatch label. | Included as a contextual field in exploratory charts. |

## Derived Columns Used in the Notebooks

These are created during analysis rather than written by the generator.

| Column | Type | Description |
|---|---|---|
| `Failure_State` | categorical | Notebook-created label used to mark rows with missing bearing temperature or vibration as sensor dropout / failure. |
| `Vibration_per_RPM` | float | Vibration normalised by RPM, used to compare roughness at different operating speeds. |
| `Vibration_per_RPM_Rolling` | float | Rolling mean of vibration per RPM for trend inspection. |
| `Day_Index` | float | Day offset from the start of the time series, used for linear trend fitting. |
| `Trend_Fit` | float | Fitted line for the WT-03 vibration-per-RPM trend. |


## Section 1 — Data Ingestion and Cleaning [15 mins]

Load the raw telemetry without modifying it first, inspect what you have, then decide what to keep, clean, or set aside.

A row with missing values might mean a failed sensor, a broken data path, or a turbine that has stopped. Discarding those rows early loses evidence. The approach here: keep all raw records in one stream for forensic review, build a separate cleaned stream for charts and statistics.

In [ ]:
# --- PARTICIPANT CODE HERE ---
# Load the telemetry CSV into a DataFrame called df.
# Run df.info() and df.isna().sum() to see the column types and where values are missing.
# The CSV is turbine_telemetry.csv in the same folder as this notebook.
# Example starting point:
# from pathlib import Path
# import numpy as np, pandas as pd
# import plotly.express as px, plotly.graph_objects as go
# from plotly.subplots import make_subplots
# csv_path = Path('turbine_telemetry.csv')
# df = pd.read_csv(csv_path, parse_dates=['Timestamp'])
# print(df.info())
# print(df.isna().sum().sort_values(ascending=False).head(10))
raise NotImplementedError('Participant task: load and inspect the telemetry data.')

In [ ]:
# --- PARTICIPANT CODE HERE ---
# Tag rows where bearing temperature or vibration data is missing as 'Sensor dropout / failure'.
# Split into two streams:
#   failure_records  — kept as-is for forensic review
#   analysis_df      — rows with complete bearing and vibration data
# Sort analysis_df by Turbine_ID then Timestamp before any rolling operations.
# Example starting point:
# df['Failure_State'] = np.where(
#     df[['Bearing_Temperature_C', 'Vibration_mm_s']].isna().any(axis=1).values,
#     'Sensor dropout / failure', 'Normal',
# )
# failure_records = df[df['Failure_State'] == 'Sensor dropout / failure'].copy()
# analysis_df = df.dropna(subset=['Bearing_Temperature_C', 'Vibration_mm_s']).copy()
# analysis_df = analysis_df.sort_values(['Turbine_ID', 'Timestamp'])
raise NotImplementedError('Participant task: tag failure rows and build two data streams.')

In [ ]:
# --- PARTICIPANT CODE HERE ---
# Remove physically impossible spikes using a 3×IQR filter on Bearing_Temperature_C and Vibration_mm_s.
# Use 3× rather than 1.5× — WT-02's real degradation signal (bearing temps up to ~80°C) falls within
# 1.5×IQR, so the tighter threshold removes the fault signal along with the sensor glitches.
# Build a single boolean mask across both columns before applying it so the quantile stats
# are calculated on the full dataset, not on a progressively smaller one.
# Example starting point:
# valid_mask = pd.Series(True, index=analysis_df.index)
# for col in ['Bearing_Temperature_C', 'Vibration_mm_s']:
#     Q1, Q3 = analysis_df[col].quantile(0.25), analysis_df[col].quantile(0.75)
#     IQR = Q3 - Q1
#     valid_mask &= (analysis_df[col] >= Q1 - 3 * IQR) & (analysis_df[col] <= Q3 + 3 * IQR)
# analysis_df = analysis_df[valid_mask]
# print(f'Rows removed: {(~valid_mask).sum()}, rows remaining: {len(analysis_df)}')
raise NotImplementedError('Participant task: apply the 3×IQR spike filter.')

In [ ]:
# --- PARTICIPANT CODE HERE ---
# Check that the cleaning kept the real degradation signal.
# Group by Turbine_ID and run describe() on Bearing_Temperature_C.
# WT-02's max should be noticeably higher than WT-01 and WT-03 if the filter was right.
raise NotImplementedError('Participant task: sanity-check the cleaned data with describe().')

## Section 2 — Interactive Anomaly Hunting via Scatter Plots [15 mins]

Plot four variables at once: vibration on y, bearing temperature on x, turbine as colour, RPM as point size. A healthy turbine should cluster with the others; a failing one will sit in a different part of that space. Use hover to check whether outlying points are windier than the rest — they won't be.

Then plot bearing temperature over time. The scatter shows what the outliers look like; the time series shows when things changed.

In [ ]:
# --- PARTICIPANT CODE HERE ---
# Build a Plotly scatter: bearing temperature on x, vibration on y, turbine as colour, RPM as point size.
# Add hover_data including wind speed, gusts, turbulence, and operational state so you can
# check weather context on any outlying point directly in the chart.
# Example starting point:
# scatter_df = analysis_df.dropna(subset=['Bearing_Temperature_C', 'Vibration_mm_s', 'RPM']).copy()
# fig = px.scatter(
#     scatter_df,
#     x='Bearing_Temperature_C', y='Vibration_mm_s',
#     color='Turbine_ID', size='RPM', size_max=6,
#     hover_data=['Timestamp', 'RPM', 'Wind_Speed_mps', 'Wind_Gust_mps',
#                 'Turbulence_Intensity', 'Operational_State'],
#     title='Vibration vs Bearing Temperature by Turbine',
# )
raise NotImplementedError('Participant task: build the bearing-temperature vs vibration scatter.')

In [ ]:
# --- PARTICIPANT CODE HERE ---
# Plot bearing temperature over time, coloured by turbine.
# Use the raw df filtered to Bearing_Temperature_C < 500 — not analysis_df.
# That keeps the real story visible: data gaps where telemetry dropped out, and the WT-02 seizure spike.
# Set connectgaps=False so a sensor dropout shows as a gap in the line, not a straight jump.
# Example starting point:
# ts_df = df[df['Bearing_Temperature_C'] < 500].copy()
# fig = px.line(ts_df, x='Timestamp', y='Bearing_Temperature_C', color='Turbine_ID',
#               hover_data=['Operational_State', 'Maintenance_Flag', 'Wind_Speed_mps'])
# fig.update_traces(connectgaps=False)
raise NotImplementedError('Participant task: build the bearing temperature time series.')

In [ ]:
# --- PARTICIPANT CODE HERE ---
# Build a four-panel chart to check the physics-based relationships that should hold for a healthy turbine:
#   top-left:     wind speed vs power          (roughly quadratic up to rated output)
#   top-right:    turbulence intensity vs vibration
#   bottom-left:  wind gust vs RPM
#   bottom-right: blade pitch vs power
# Colour each turbine separately and add a polynomial fit line to each panel.
# If a turbine breaks one relationship while obeying the others, that is the starting point for diagnosis.
# Use make_subplots(rows=2, cols=2) from plotly.subplots.
# The columns you need: Wind_Speed_mps, Power_kW, Vibration_mm_s, RPM,
#                       Wind_Gust_mps, Turbulence_Intensity, Blade_Pitch_Deg
raise NotImplementedError('Participant task: build the four-panel physics-relationship chart.')

### Interactive Challenge

Use zoom and hover to find the exact date WT-02's bearing temperature spikes and the first date WT-03 rises above WT-01. Both dates are useful in Section 4.

## Section 3 — Vibration Normalised by RPM [15 mins]

Raw vibration rises with RPM even in a perfectly healthy turbine — a faster-spinning rotor produces more vibration. Dividing by RPM asks a different question: how much vibration per rotation?

A healthy bearing gives a stable ratio. A degrading one produces more vibration per rotation as surfaces get rougher. That ratio is the early warning signal.

One practical issue: when RPM is near zero (turbine below cut-in wind speed), dividing inflates the ratio into noise. Replace zero RPM with NaN before dividing — pandas skips NaN in rolling calculations.

In [ ]:
# --- PARTICIPANT CODE HERE ---
# Compute Vibration_per_RPM for WT-03 from analysis_df.
# Replace zero RPM with NaN before dividing.
# Add a 4-hour rolling mean (window=24, min_periods=6 — each step is 10 minutes).
# Plot both the raw ratio and the rolling mean on the same figure using fig.add_scatter().
# Example starting point:
# wt03_df = analysis_df[analysis_df['Turbine_ID'] == 'WT-03'].sort_values('Timestamp').copy()
# wt03_df['Vibration_per_RPM'] = wt03_df['Vibration_mm_s'] / wt03_df['RPM'].replace(0, np.nan)
# wt03_df['Vibration_per_RPM_Rolling'] = wt03_df['Vibration_per_RPM'].rolling(window=24, min_periods=6).mean()
raise NotImplementedError('Participant task: compute and plot the WT-03 vibration/RPM ratio.')

In [ ]:
# --- PARTICIPANT CODE HERE ---
# Add WT-01's 4-hour rolling mean to the same figure as a healthy baseline.
# If WT-03's ratio rises while WT-01's stays flat, the trend is an asset signal, not a weather effect.
# Example starting point:
# wt01_df = analysis_df[analysis_df['Turbine_ID'] == 'WT-01'].sort_values('Timestamp').copy()
# wt01_df['Vibration_per_RPM'] = wt01_df['Vibration_mm_s'] / wt01_df['RPM'].replace(0, np.nan)
# wt01_df['Vibration_per_RPM_Rolling'] = wt01_df['Vibration_per_RPM'].rolling(window=24, min_periods=6).mean()
# fig.add_scatter(x=wt01_df['Timestamp'], y=wt01_df['Vibration_per_RPM_Rolling'],
#                 mode='lines', name='WT-01 rolling mean (healthy baseline)',
#                 line=dict(dash='dot', width=2))
raise NotImplementedError('Participant task: add the WT-01 healthy baseline to the figure.')

## Section 4 — Operational Decision and Dashboard Ethics [15 mins]

One maintenance slot, three turbines. WT-01 healthy. WT-02 already offline. WT-03 drifting.

Before writing code, think through the decision: spend the budget repairing the known failure (WT-02) or preventing the predicted one (WT-03)?

Build the evidence in four steps:
1. Set a warning threshold from WT-01's healthy behaviour (mean + 2 standard deviations).
2. Fit a linear trend to WT-03's vibration/RPM and project when it crosses that threshold.
3. Show the trend, the threshold, and the power output comparison in a two-panel dashboard.
4. Summarise each turbine's status in a table and state the recommendation.

In [ ]:
# --- PARTICIPANT CODE HERE ---
# Add Vibration_per_RPM to analysis_df for all three turbines.
# Calculate the warning threshold from WT-01's Vibration_per_RPM: mean + 2 standard deviations.
# Mean + 2 SD covers ~95% of WT-01's normal behaviour — anything above it is unusual enough to flag.
# Example starting point:
# analysis_df['Vibration_per_RPM'] = analysis_df['Vibration_mm_s'] / analysis_df['RPM'].replace(0, np.nan)
# healthy_baseline = analysis_df.loc[analysis_df['Turbine_ID'] == 'WT-01', 'Vibration_per_RPM'].dropna()
# warning_threshold = healthy_baseline.mean() + 2 * healthy_baseline.std()
# print(f'Warning threshold (mean + 2 SD): {warning_threshold:.4f}')
raise NotImplementedError('Participant task: calculate the warning threshold from WT-01.')

In [ ]:
# --- PARTICIPANT CODE HERE ---
# Fit a linear trend to WT-03's Vibration_per_RPM over time.
# Convert timestamps to a Day_Index (elapsed days from the start of the series) for the fit.
# If the slope is positive, project the day the trend line will cross warning_threshold.
# Example starting point:
# wt03_ratio = analysis_df.loc[analysis_df['Turbine_ID'] == 'WT-03',
#     ['Timestamp', 'Vibration_per_RPM', 'Wind_Speed_mps', 'Wind_Gust_mps',
#      'Turbulence_Intensity', 'Bearing_Temperature_C']].dropna().copy()
# wt03_ratio['Day_Index'] = (wt03_ratio['Timestamp'] - wt03_ratio['Timestamp'].min()).dt.total_seconds() / 86400
# ratio_coeffs = np.polyfit(wt03_ratio['Day_Index'], wt03_ratio['Vibration_per_RPM'], deg=1)
# wt03_ratio['Trend_Fit'] = np.polyval(ratio_coeffs, wt03_ratio['Day_Index'])
# if ratio_coeffs[0] > 0:
#     projected_day = (warning_threshold - ratio_coeffs[1]) / ratio_coeffs[0]
#     projected_date = wt03_ratio['Timestamp'].min() + pd.to_timedelta(projected_day, unit='D')
#     print(f'Projected threshold crossing: {projected_date.strftime("%Y-%m-%d")}')
raise NotImplementedError('Participant task: fit the trend and project the threshold crossing.')

In [ ]:
# --- PARTICIPANT CODE HERE ---
# Build a two-panel evidence dashboard using make_subplots(rows=1, cols=2):
#   Left panel:  WT-03 Vibration_per_RPM scatter + trend line +
#                horizontal warning threshold (add_hline) +
#                vertical projected crossing date (add_vline)
#   Right panel: power output vs wind speed for all three turbines with quadratic fit lines
raise NotImplementedError('Participant task: build the two-panel evidence dashboard.')

In [ ]:
# --- PARTICIPANT CODE HERE ---
# Print a summary table with one row per turbine:
#   mean Power_kW, max Vibration_per_RPM, latest Maintenance_Flag, latest Operational_State
# Then print the projected threshold crossing date for WT-03 and state the maintenance recommendation.
raise NotImplementedError('Participant task: print the summary table and maintenance recommendation.')

### Ethics and Governance Discussion

No more code — this is a whole-group discussion.

WT-02 is a sunk cost. Its revenue is already lost. Fixing it returns to baseline. Preventing WT-03 from failing avoids a future outage and typically a more expensive emergency repair. The data points to WT-03; the projected crossing date is the deadline.

A few things the data alone can't settle:
- If WT-03 seizes while the decision is being made and a technician is nearby — who is responsible? The trend was already visible.
- A seized bearing at speed can release lubricating oil. How does that change the cost calculation beyond lost revenue?
- This dashboard shows the trend but not its uncertainty. If the fit is poor, the projection could be off by weeks. How do you communicate that to a manager without burying the chart in caveats?
- Rows were removed, a threshold was set, failure records were kept separate. None of that is visible in the final chart. Should someone using it to make a budget decision know about those choices?

## Wrap-Up

By the end of this notebook you should be able to:

1. Load and inspect messy telemetry and say what the missing values mean before touching them
2. Split into forensic and statistical streams and explain why forward-filling is the wrong default for sensor dropout
3. Build four relationship scatter plots and read them for signs a turbine is drifting from expected physics
4. Normalise vibration by RPM to separate speed effects from bearing roughness, and use a rolling mean to surface slow trends
5. Fit a linear trend, set a threshold from a healthy baseline, and project a crossing date
6. State the maintenance recommendation in terms the data supports — and name at least two things the data can't settle